### 문제 3-1 :  콘텐츠분쟁해결 RAG 시스템 - 간단 실습 가이드

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── 0단계: 문서 로드 ──────────────────────────
loader = PyPDFLoader('../data/콘텐츠분쟁해결_사례.pdf')

docs = loader.load()

print(f"로드된 페이지 수: {len(docs)}")
print(docs[0].metadata)          # 어떤 정보가 붙어있는지 확인

# ── 1단계: 문서 분할 ──────────────────────────
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,        #  법률 사례는 1200-1800자 권장
    chunk_overlap=300,      #  사례 맥락 보존을 위해 200-400자
    separators=[
        "\n【사건개요】",     #  법률 문서 섹션 구분자
        "\n【쟁점사항】",     #  쟁점 부분 구분
        "\n【처리경위】",     #  처리 과정 구분
        "\n【처리결과】",     #  결과 부분 구분
        "\n■", "\n\n", "\n", ".", " ", ""
    ]
)

chunks = text_splitter.split_documents(docs)

print(f"분할된 청크 수: {len(chunks)}")
print(chunks[0].page_content[:300])   # 첫 청크가 사례 섹션 단위로 잘렸는지 눈으로 확인

In [ ]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()   # .env의 UPSTAGE_API_KEY 불러오기

# ── 2단계: 임베딩 모델 설정 ──────────────────────
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

# ── 벡터DB 생성 (3단계 retriever가 쓸 vectorstore) ──
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"저장된 벡터 수: {vectorstore.index.ntotal}")   # 청크 수와 같아야 정상

In [ ]:
# ── 3단계: 검색기 설정 ──────────────────────────
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# ── 검색 테스트 (체인 만들기 전, 검색기만 단독 확인) ──
test_query = "게임 아이템이 사라졌는데 복구를 거부합니다"
results = retriever.invoke(test_query)

print(f"검색된 문서 수: {len(results)}")          
for i, doc in enumerate(results, 1):
    print(f"\n[{i}] 페이지 {doc.metadata.get('page')}")
    print(doc.page_content[:150])                

In [ ]:
from langchain_upstage import ChatUpstage
from langchain_core.prompts import ChatPromptTemplate

# ── 4단계: LLM 설정 ─────────────────────────────
llm = ChatUpstage(
    model="solar-pro3",                 
    base_url="https://api.upstage.ai/v1",
    temperature=0.2 
)

# ── 5단계: 법률 자문 프롬프트 ───────────────────
prompt_template = """
당신은 콘텐츠 분야 전문 법률 자문가입니다. 
아래 분쟁조정 사례들을 바탕으로 정확하고 전문적인 법률 조언을 제공해주세요.

관련 분쟁사례:
{context}

상담 내용: {question}

답변 가이드라인:
1. 제시된 사례들을 근거로 답변하세요                    
2. 관련 법령이나 조항이 있다면 명시하세요               
3. 비슷한 사례의 처리경위와 결과를 참고하여 설명하세요   
4. 실무적 해결방안을 단계별로 제시하세요               
5. 사례에 없는 내용은 "제시된 사례집에서는 확인할 수 없습니다"라고 명시하세요  

전문 법률 조언:
"""
prompt = ChatPromptTemplate.from_template(prompt_template)

# ── 확인: 빈칸 이름이 제대로 인식됐는지 ──────────
print(prompt.input_variables) 

['context', 'question']


In [6]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ── 검색 결과 → 문자열 변환기 ──────────────────
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# ── 6단계: LCEL 체인 ──────────────────────────
qa_chain = (
    {
        "context": retriever | format_docs,          # TODO: 검색 → 문자열 변환 (위 그림 윗줄)
        "question": RunnablePassthrough()              # TODO: 질문을 그대로 통과 (위 그림 아랫줄)
    }
    | prompt                              # TODO: 빈칸 채우기
    | llm                              # TODO: 답변 생성
    | StrOutputParser()
)

# ── 단일 질문 테스트 ──────────────────────────
answer = qa_chain.invoke("온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?")
print(answer)

# 온라인 게임 시스템 오류로 인한 아이템 복구 관련 법률 조언

## 1. 사례 기반 분석

제시된 분쟁조정 사례들을 종합적으로 분석하면 다음과 같은 핵심 사항들을 확인할 수 있습니다:

- **2006년 사례**: 프로그램 오류로 인한 아이템 소멸 시, 게임회사가 복구 가능성을 인정하고 실제로 복구 처리한 사례
- **2009년 사례**: 시스템 오류 발생 여부가 명확하지 않은 경우(렉 현상 등), 게임회사의 책임이 제한되거나 통신회선/PC 오류 등 다른 원인을 제시한 사례
- **2006년 다른 사례**: 프로그램 오류로 인한 아이템 소멸 시, 게임회사가 복구 처리하기로 결정한 사례

## 2. 법적 근거

### 가. 민법 관련 조항
- **민법 제250조(도품, 유실물에 대한 특례)**: 도품이나 유실물은 2년 내에 반환 청구 가능
- **민법 제251조**: 도품/유실물을 선의로 매수한 경우, 대가를 변상하고 반환 청구 가능

### 나. 게임 이용약관 관련 조항
- **계정 소유권 및 거래 제한**: 대부분의 게임회사는 현금거래 및 계정공유를 인정하지 않음
- **서비스 중단 및 오류에 대한 책임**: 시스템 오류 발생 시 책임 범위 명시

## 3. 판례 참조 및 처리경위

### 가. 복구가 인정된 사례 (2006년)
- **처리경위**: 게임회사가 프로그램 오류로 인한 아이템 소멸을 인정하고 복구 처리
- **결과**: 신청인이 게임 캐릭터를 복구 받음

### 나. 복구가 거절된 사례 (2009년)
- **처리경위**: 시스템 오류 발생 여부가 명확하지 않고, 신청인의 사용 기록에 문제가 없음
- **결과**: 통신회선 불안정 또는 PC 오류로 판단하여 복구 거절

### 다. 계정 소유권 관련 사례
- **처리경위**: 계정 명의자가 아닌 경우 복구 처리 불가
- **결과**: 계정 명의자에게 직접 복구 신청 요구

## 4. 실무적 해결방안 (단계별)

### 1단계: 증거 수집
- **시스템 오류 발생 시점 기록**: 스크린샷, 로그 